# Monthly Forecasting: Seasonal Naive vs SARIMAX

**Type:** Original supplementary practice  
**Data:** Synthetic monthly series

A chronological 12-month holdout is used to compare a simple seasonal-naive forecast with SARIMAX. This demonstrates model benchmarking; the synthetic data do not provide evidence for real demand forecasting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

rng = np.random.default_rng(119)
n = 120
dates = pd.date_range("2015-01-01", periods=n, freq="MS")
t = np.arange(n)
trend = .25*t
season = 10*np.sin(2*np.pi*t/12) + 3*np.cos(2*np.pi*t/12)
noise = rng.normal(0,2.2,n)
series = pd.Series(150+trend+season+noise, index=dates, name="synthetic_demand")
horizon = 12
train, test = series.iloc[:-horizon], series.iloc[-horizon:]
print("Train:", train.index.min(), "to", train.index.max())
print("Test:", test.index.min(), "to", test.index.max())

## 1. Seasonal-naive benchmark

In [ ]:
seasonal_naive = pd.Series(
    [train.iloc[-12+i] for i in range(horizon)],
    index=test.index, name="seasonal_naive"
)

## 2. Fit SARIMAX

In [ ]:
model = SARIMAX(
    train,
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
fitted = model.fit(disp=False)
sarimax_forecast = fitted.get_forecast(steps=horizon).predicted_mean
sarimax_forecast.index = test.index

## 3. Evaluate using the same holdout

In [ ]:
def metrics(actual, predicted, label):
    return {
        "model":label,
        "MAE":mean_absolute_error(actual,predicted),
        "RMSE":np.sqrt(mean_squared_error(actual,predicted))
    }
results = pd.DataFrame([
    metrics(test, seasonal_naive, "Seasonal naive"),
    metrics(test, sarimax_forecast, "SARIMAX")
]).set_index("model")
display(results.round(3))

## 4. Visual inspection

In [ ]:
fig, ax = plt.subplots(figsize=(11,4.5))
ax.plot(train.index, train, label="Training data")
ax.plot(test.index, test, label="Actual holdout", linewidth=2)
ax.plot(test.index, seasonal_naive, label="Seasonal naive")
ax.plot(test.index, sarimax_forecast, label="SARIMAX")
ax.set(title="Synthetic monthly series: 12-month forecast", xlabel="Month", ylabel="Value")
ax.legend()
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## Further analysis

Inspect residual autocorrelation, test stationarity and differencing assumptions, and use rolling-origin validation across multiple origins. Compare other orders only within a training/validation process; do not repeatedly tune against the final holdout. Report prediction intervals and domain-specific forecast error costs for applied work.